# MARL DQN Training & Evaluation Curves

This notebook visualises two distinct phases:

### Part A \u2014 Training (200 episodes, Vancouver)
Metrics recorded **during training** in the model checkpoint: **reward, loss, epsilon, episode length**.
Training disables SUMO emission output for speed, so CO\u2082 and throughput are **not available** during training.

### Part B \u2014 Evaluation (120 episodes, Cologne)
The trained model is evaluated on the **unseen Cologne** network with full metric collection enabled.
This is where **reward, CO\u2082 emissions, and throughput** are all available per episode.

> **Why 200 vs 120?** Training ran for 200 episodes on Vancouver. Evaluation ran for 120 episodes on Cologne (a separate command). These are different phases with different episode counts.

**Model:** `marl_vancouver_shared_dqn_regionaware_v2.pt`

**Training command:**
```
python train_marl_los_angeles.py
  --dataset vancouver --episodes 200 --duration 1200
  --decision-interval 5 --controlled-lights-ratio 0.5
  --lanes-per-tl 8 --regional-reward-weight 0.01
  --region-grid-size 500 --seed 42 --save-every 10
  --out models/marl_vancouver_shared_dqn_regionaware_v2.pt
```

**Charts produced:**
1. Training Reward Curve (200 episodes)
2. Training Loss Curve
3. Epsilon Decay Schedule
4. Combined Training Dashboard (4-panel)
5. Training Convergence Analysis
6. Evaluation Reward vs Episodes (Cologne, 120 episodes)
7. Evaluation CO\u2082 Emissions vs Episodes
8. Evaluation Throughput vs Episodes
9. Combined Evaluation Dashboard (Reward, Emissions, Throughput)
10. Reward\u2013Emissions\u2013Throughput Correlation

In [24]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter

ROOT = Path(".").resolve()
MODEL_PATH = ROOT / "models" / "marl_cologne_shared_dqn_regionaware_v9_scale07.pt"
EVAL_PATH = ROOT / "output" / "eval_cologne_MARL_DQN_v9_scale07_ep120_seed42.json"
FIG_DIR = ROOT / "Report" / "figures" / "training"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.facecolor": "white",
})
sns.set_theme(style="whitegrid", palette="colorblind")

COLOR_PRIMARY = "#E63946"
COLOR_SECONDARY = "#457B9D"
COLOR_ACCENT = "#2A9D8F"
COLOR_WARN = "#F4A261"
COLOR_SMOOTH = "#1D3557"


def save_fig(stem: str):
    png = FIG_DIR / f"{stem}.png"
    pdf = FIG_DIR / f"{stem}.pdf"
    plt.savefig(png, dpi=300, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {png.name}, {pdf.name}")


print("ROOT:", ROOT)
print("Model:", MODEL_PATH)
print("Eval:", EVAL_PATH)
print("Figures will be saved to:", FIG_DIR)

ROOT: D:\Final Year Project\traffic-signal-control
Model: D:\Final Year Project\traffic-signal-control\models\marl_cologne_shared_dqn_regionaware_v8_scale07.pt
Eval: D:\Final Year Project\traffic-signal-control\output\eval_cologne_MARL_DQN_v8_ep300_seed42.json
Figures will be saved to: D:\Final Year Project\traffic-signal-control\Report\figures\training


## 1. Load Training History from Model Checkpoint

The `.pt` checkpoint stores a `training_history` dict with per-episode rewards, episode lengths, per-step losses, epsilon values, and Q-values recorded during the 200-episode training run.

In [25]:
checkpoint = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)

history = checkpoint.get("training_history", {})
config = checkpoint.get("config", {})

episode_rewards = np.array(history.get("episode_rewards", []), dtype=float)
episode_lengths = np.array(history.get("episode_lengths", []), dtype=float)
losses = np.array(history.get("losses", []), dtype=float)
epsilon_values = np.array(history.get("epsilon_values", []), dtype=float)
q_values = np.array(history.get("q_values", []), dtype=float)

n_episodes = len(episode_rewards)
episodes = np.arange(1, n_episodes + 1)

print(f"Training episodes: {n_episodes}")
print(f"Total training steps (losses recorded): {len(losses):,}")
print(f"Epsilon values recorded: {len(epsilon_values):,}")
print(f"Q-values recorded: {len(q_values):,}")
print(f"\nCheckpoint config:")
for k, v in config.items():
    if k not in ("q_network_state_dict", "target_network_state_dict", "optimizer_state_dict"):
        print(f"  {k}: {v}")
print(f"\nFinal epsilon: {checkpoint.get('epsilon', 'N/A')}")
print(f"Total step count: {checkpoint.get('step_count', 'N/A'):,}")

Training episodes: 300
Total training steps (losses recorded): 71,997
Epsilon values recorded: 71,997
Q-values recorded: 0

Checkpoint config:
  state_dim: 77
  action_dim: 2
  lr: 5e-05
  gamma: 0.9
  epsilon_start: 1.0
  epsilon_end: 0.05
  epsilon_decay: 1.0
  batch_size: 128
  memory_size: 200000
  target_update_freq: 2000
  hidden_dims: [256, 256, 128]
  device: cpu
  double_dqn: True
  dueling: True
  n_step: 1
  tau: 0.005
  max_green_phases: 6
  dataset: cologne
  lanes_per_tl: 20
  decision_interval: 5
  max_controlled_lights: 3
  controlled_lights_ratio: 0.5
  regional_reward_weight: 0.01
  region_grid_size: 500.0

Final epsilon: 0.050000000000000044
Total step count: 2,880,000


## 2. Load Evaluation Data (Cologne, 120 episodes)

The evaluation JSON contains per-episode metrics including reward, CO\u2082 emissions, throughput, waiting time, queue length, and more.

In [26]:
with open(EVAL_PATH, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

eval_results = eval_data.get("results", [])
eval_df = pd.DataFrame([
    {
        "episode": r["episode"],
        "reward_sum": r.get("reward_sum"),
        "avg_reward": r.get("avg_reward"),
        **r.get("metrics", {}),
    }
    for r in eval_results
]).sort_values("episode").reset_index(drop=True)

for col in eval_df.columns:
    if col not in ("episode",):
        eval_df[col] = pd.to_numeric(eval_df[col], errors="coerce")

print(f"Evaluation episodes loaded: {len(eval_df)}")
print(f"Dataset: {eval_data.get('dataset', 'N/A')}")
print(f"Available metrics: {sorted([c for c in eval_df.columns if c != 'episode'])}")
print(f"\nQuick summary:")
for col in ["reward_sum", "total_co2", "throughput_per_hour"]:
    if col in eval_df.columns:
        vals = eval_df[col].dropna()
        print(f"  {col}: mean={vals.mean():.2f}, std={vals.std():.2f}, min={vals.min():.2f}, max={vals.max():.2f}")

Evaluation episodes loaded: 120
Dataset: cologne
Available metrics: ['acceleration_std', 'avg_acceleration', 'avg_co2_per_vehicle', 'avg_fuel_per_vehicle', 'avg_lane_occupancy', 'avg_max_waiting_time_per_vehicle', 'avg_pressure', 'avg_queue_length', 'avg_reward', 'avg_speed', 'avg_waiting_time', 'co2_per_km', 'congestion_index', 'departed_per_hour', 'harsh_braking_events', 'lane_occupancy_std', 'max_lane_occupancy', 'max_pressure', 'max_queue_length', 'max_queue_length_vehicles', 'max_speed', 'max_waiting_time', 'percentage_vehicles_waited', 'reward_sum', 'throughput', 'throughput_per_hour', 'total_co2', 'total_fuel', 'total_nox', 'total_pmx', 'total_waiting_time', 'vehicle_type_stats', 'vehicles_arrived', 'vehicles_departed', 'vehicles_seen', 'vehicles_total', 'vehicles_with_waiting']

Quick summary:
  reward_sum: mean=-2577.37, std=49.65, min=-2677.26, max=-2445.58
  total_co2: mean=0.00, std=0.00, min=0.00, max=0.00
  throughput_per_hour: mean=517.08, std=33.90, min=441.00, max=606.

---
## 3. Training Reward Curve (Episodes 1–200)

Shows how the agent's cumulative reward per episode evolves during training on the Vancouver network. A rising trend indicates the agent is learning a better traffic signal policy.

In [27]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(episodes, episode_rewards, alpha=0.25, color=COLOR_PRIMARY, linewidth=0.8, label="Per-Episode Reward")

window = min(20, max(1, n_episodes // 10))
ma = pd.Series(episode_rewards).rolling(window=window, min_periods=1).mean().values
ax.plot(episodes, ma, color=COLOR_SMOOTH, linewidth=2.5, label=f"{window}-Episode Moving Average")

overall_mean = np.mean(episode_rewards)
ax.axhline(y=overall_mean, color="gray", linestyle="--", alpha=0.5, linewidth=1,
           label=f"Overall Mean ({overall_mean:.1f})")

best_idx = np.argmax(episode_rewards)
ax.scatter([episodes[best_idx]], [episode_rewards[best_idx]], color="gold", edgecolors="black",
           s=100, zorder=5, label=f"Best: {episode_rewards[best_idx]:.1f} @ ep {episodes[best_idx]}")

ax.fill_between(episodes, episode_rewards, ma, alpha=0.07, color=COLOR_PRIMARY)

ax.set_xlabel("Episode")
ax.set_ylabel("Average Reward per Step")
ax.set_title("MARL DQN Training Reward Curve (Vancouver, 200 Episodes)")
ax.legend(loc="lower right", framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim(1, n_episodes)
save_fig("training_reward_curve")

  Saved: training_reward_curve.png, training_reward_curve.pdf


## 4. Training Loss Curve

The DQN loss (Huber loss between predicted and target Q-values). In DQN with experience replay, a **moderate increase** in loss during early-to-mid training is normal and expected \u2014 it reflects the replay buffer filling with more diverse experiences and Q-value targets shifting as the network learns. What matters is that the loss remains **bounded** (not diverging to infinity) and the reward improves.

In [28]:
if len(losses) > 0:
    steps_per_ep = len(losses) / n_episodes if n_episodes > 0 else len(losses)
    ep_losses = []
    for i in range(n_episodes):
        start = int(i * steps_per_ep)
        end = int((i + 1) * steps_per_ep)
        chunk = losses[start:end]
        ep_losses.append(np.mean(chunk) if len(chunk) > 0 else np.nan)
    ep_losses = np.array(ep_losses)

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(episodes, ep_losses, alpha=0.25, color=COLOR_SECONDARY, linewidth=0.8, label="Per-Episode Avg Loss")

    ma_loss = pd.Series(ep_losses).rolling(window=window, min_periods=1).mean().values
    ax.plot(episodes, ma_loss, color=COLOR_SMOOTH, linewidth=2.5, label=f"{window}-Episode Moving Average")

    ax.set_xlabel("Episode")
    ax.set_ylabel("Average Loss (MSE)")
    ax.set_title("MARL DQN Training Loss Curve (Vancouver, 200 Episodes)")
    ax.legend(loc="upper right", framealpha=0.9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(1, n_episodes)
    save_fig("training_loss_curve")
else:
    print("No loss data recorded in checkpoint.")

  Saved: training_loss_curve.png, training_loss_curve.pdf


## 5. Epsilon Decay Schedule

Shows the exploration–exploitation trade-off during training. Epsilon starts high (more exploration) and decays towards a minimum value (more exploitation of learned policy).

In [29]:
if len(epsilon_values) > 0:
    eps_per_ep = len(epsilon_values) / n_episodes if n_episodes > 0 else len(epsilon_values)
    ep_epsilon = []
    for i in range(n_episodes):
        start = int(i * eps_per_ep)
        end = int((i + 1) * eps_per_ep)
        chunk = epsilon_values[start:end]
        ep_epsilon.append(chunk[-1] if len(chunk) > 0 else np.nan)
    ep_epsilon = np.array(ep_epsilon)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(episodes, ep_epsilon, color=COLOR_ACCENT, linewidth=2.5)
    ax.fill_between(episodes, 0, ep_epsilon, alpha=0.15, color=COLOR_ACCENT)

    ax.set_xlabel("Episode")
    ax.set_ylabel("Epsilon (\u03b5)")
    ax.set_title("Exploration Rate (\u03b5) Decay During Training")
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(1, n_episodes)

    ax.annotate(f"Start: {ep_epsilon[0]:.3f}", xy=(1, ep_epsilon[0]),
                fontsize=10, color=COLOR_ACCENT, fontweight="bold",
                xytext=(20, 10), textcoords="offset points",
                arrowprops=dict(arrowstyle="->", color=COLOR_ACCENT))
    ax.annotate(f"End: {ep_epsilon[-1]:.3f}", xy=(n_episodes, ep_epsilon[-1]),
                fontsize=10, color=COLOR_ACCENT, fontweight="bold",
                xytext=(-80, 20), textcoords="offset points",
                arrowprops=dict(arrowstyle="->", color=COLOR_ACCENT))

    save_fig("training_epsilon_decay")
else:
    print("No epsilon data recorded in checkpoint.")

  Saved: training_epsilon_decay.png, training_epsilon_decay.pdf


## 6. Combined Training Dashboard (4-Panel)

A single figure showing all key training metrics: reward, loss, epsilon, and episode length.

In [30]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- Panel (a): Reward ---
ax = axes[0, 0]
ax.plot(episodes, episode_rewards, alpha=0.2, color=COLOR_PRIMARY, linewidth=0.8)
ma_r = pd.Series(episode_rewards).rolling(window=window, min_periods=1).mean().values
ax.plot(episodes, ma_r, color=COLOR_PRIMARY, linewidth=2.5, label=f"MA-{window}")
ax.set_xlabel("Episode")
ax.set_ylabel("Avg Reward / Step")
ax.set_title("(a) Training Reward")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)

# --- Panel (b): Loss ---
ax = axes[0, 1]
if len(losses) > 0:
    ax.plot(episodes, ep_losses, alpha=0.2, color=COLOR_SECONDARY, linewidth=0.8)
    ma_l = pd.Series(ep_losses).rolling(window=window, min_periods=1).mean().values
    ax.plot(episodes, ma_l, color=COLOR_SECONDARY, linewidth=2.5, label=f"MA-{window}")
    ax.set_ylabel("Avg Loss")
    ax.legend(loc="upper right")
else:
    ax.text(0.5, 0.5, "No loss data", transform=ax.transAxes, ha="center", fontsize=14)
ax.set_xlabel("Episode")
ax.set_title("(b) Training Loss")
ax.grid(True, alpha=0.3)

# --- Panel (c): Epsilon ---
ax = axes[1, 0]
if len(epsilon_values) > 0:
    ax.plot(episodes, ep_epsilon, color=COLOR_ACCENT, linewidth=2.5)
    ax.fill_between(episodes, 0, ep_epsilon, alpha=0.1, color=COLOR_ACCENT)
    ax.set_ylabel("Epsilon (\u03b5)")
    ax.set_ylim(bottom=0)
else:
    ax.text(0.5, 0.5, "No epsilon data", transform=ax.transAxes, ha="center", fontsize=14)
ax.set_xlabel("Episode")
ax.set_title("(c) Exploration Rate")
ax.grid(True, alpha=0.3)

# --- Panel (d): Episode Length ---
ax = axes[1, 1]
ax.plot(episodes, episode_lengths, alpha=0.3, color=COLOR_WARN, linewidth=0.8)
ma_len = pd.Series(episode_lengths).rolling(window=window, min_periods=1).mean().values
ax.plot(episodes, ma_len, color=COLOR_WARN, linewidth=2.5, label=f"MA-{window}")
ax.set_xlabel("Episode")
ax.set_ylabel("Steps per Episode")
ax.set_title("(d) Episode Length")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)

fig.suptitle("MARL DQN Training Dashboard (Vancouver, 200 Episodes)", fontsize=16, y=1.02)
fig.tight_layout()
save_fig("training_dashboard_4panel")

  Saved: training_dashboard_4panel.png, training_dashboard_4panel.pdf


## 7. Training Convergence Analysis

We check whether the agent has converged by comparing the first and last quarters of training, and by examining the cumulative mean and rolling standard deviation of rewards.

In [31]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Cumulative mean
ax = axes[0]
cum_mean = pd.Series(episode_rewards).expanding().mean().values
ax.plot(episodes, cum_mean, color=COLOR_PRIMARY, linewidth=2.5)
ax.axhline(y=cum_mean[-1], color="gray", linestyle="--", alpha=0.5,
           label=f"Final: {cum_mean[-1]:.1f}")
ax.set_xlabel("Episode")
ax.set_ylabel("Cumulative Mean Reward / Step")
ax.set_title("(a) Cumulative Mean Convergence")
ax.legend()
ax.grid(True, alpha=0.3)

# (b) Rolling std
ax = axes[1]
rolling_std = pd.Series(episode_rewards).rolling(window=20, min_periods=5).std().values
ax.plot(episodes, rolling_std, color=COLOR_SECONDARY, linewidth=2.5)
ax.set_xlabel("Episode")
ax.set_ylabel("Rolling Std Dev (20-ep window)")
ax.set_title("(b) Reward Stability")
ax.grid(True, alpha=0.3)

# (c) First quarter vs last quarter
ax = axes[2]
q_size = n_episodes // 4
quarters = {
    f"Ep 1\u2013{q_size}": episode_rewards[:q_size],
    f"Ep {q_size+1}\u2013{2*q_size}": episode_rewards[q_size:2*q_size],
    f"Ep {2*q_size+1}\u2013{3*q_size}": episode_rewards[2*q_size:3*q_size],
    f"Ep {3*q_size+1}\u2013{n_episodes}": episode_rewards[3*q_size:],
}
labels = list(quarters.keys())
means = [np.mean(v) for v in quarters.values()]
stds = [np.std(v) for v in quarters.values()]
colors_bar = [COLOR_SECONDARY, COLOR_ACCENT, COLOR_WARN, COLOR_PRIMARY]
x = np.arange(len(labels))
bars = ax.bar(x, means, yerr=stds, capsize=6, color=colors_bar,
              edgecolor="black", linewidth=0.5, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right")
ax.set_ylabel("Mean Reward")
ax.set_title("(c) Reward by Training Quarter")
ax.grid(True, alpha=0.2, axis="y")
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + abs(m) * 0.02, f"{m:.1f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

fig.suptitle("Training Convergence Analysis", fontsize=15, y=1.02)
fig.tight_layout()
save_fig("training_convergence_analysis")

first_q = episode_rewards[:q_size]
last_q = episode_rewards[-q_size:]
improvement = (np.mean(last_q) - np.mean(first_q)) / abs(np.mean(first_q)) * 100
print(f"\nTraining reward summary:")
print(f"  First quarter (ep 1\u2013{q_size}):  mean={np.mean(first_q):.1f}, std={np.std(first_q):.1f}")
print(f"  Last quarter  (ep {3*q_size+1}\u2013{n_episodes}): mean={np.mean(last_q):.1f}, std={np.std(last_q):.1f}")
print(f"  Improvement: {improvement:+.1f}%")

  Saved: training_convergence_analysis.png, training_convergence_analysis.pdf

Training reward summary:
  First quarter (ep 1–75):  mean=-0.3, std=0.0
  Last quarter  (ep 226–300): mean=-0.2, std=0.0
  Improvement: +2.5%


---
## 8. Evaluation: Reward vs Episodes (Cologne, 120 Episodes)

The trained model is evaluated on the unseen Cologne network. Each episode uses a different SUMO seed, so variation reflects generalization under stochastic traffic.

In [32]:
def plot_eval_metric(df, col, ylabel, title, figname, higher_better=True, scale=1.0, unit_label=""):
    if col not in df.columns:
        print(f"Column '{col}' not found.")
        return
    y = df[col].dropna()
    if len(y) == 0:
        print(f"No valid data for '{col}'.")
        return

    eps = df.loc[y.index, "episode"].values
    yv = y.values / scale

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(eps, yv, alpha=0.25, color=COLOR_PRIMARY, linewidth=0.8, label="Per-Episode")

    w = min(15, max(1, len(yv) // 8))
    ma = pd.Series(yv).rolling(window=w, min_periods=1).mean().values
    ax.plot(eps, ma, color=COLOR_SMOOTH, linewidth=2.5, label=f"{w}-Episode Moving Average")
    ax.fill_between(eps, yv, ma, alpha=0.07, color=COLOR_PRIMARY)

    mu = np.mean(yv)
    ax.axhline(y=mu, color="gray", linestyle="--", alpha=0.5, linewidth=1,
               label=f"Mean: {mu:.2f}{unit_label}")

    best_idx = np.argmax(yv) if higher_better else np.argmin(yv)
    marker_label = "Best" if higher_better else "Lowest"
    ax.scatter([eps[best_idx]], [yv[best_idx]], color="gold", edgecolors="black",
               s=100, zorder=5, label=f"{marker_label}: {yv[best_idx]:.2f} @ ep {int(eps[best_idx])}")

    ax.set_xlabel("Episode")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="best", framealpha=0.9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(eps[0], eps[-1])
    save_fig(figname)

    print(f"  {col}: mean={mu:.2f}, std={np.std(yv):.2f}, min={np.min(yv):.2f}, max={np.max(yv):.2f}")


plot_eval_metric(
    eval_df, "reward_sum",
    ylabel="Cumulative Reward",
    title="MARL DQN Evaluation Reward vs Episode (Cologne, 120 Episodes)",
    figname="eval_reward_vs_episode",
    higher_better=True,
)

  Saved: eval_reward_vs_episode.png, eval_reward_vs_episode.pdf
  reward_sum: mean=-2577.37, std=49.45, min=-2677.26, max=-2445.58


## 9. Evaluation: CO\u2082 Emissions vs Episodes

Total CO\u2082 emissions per episode during evaluation. Lower is better — indicates the agent's policy produces less environmental impact.

In [33]:
plot_eval_metric(
    eval_df, "total_co2",
    ylabel="Total CO\u2082 Emissions (\u00d710\u2079 mg)",
    title="MARL DQN Evaluation CO\u2082 Emissions vs Episode (Cologne, 120 Episodes)",
    figname="eval_co2_vs_episode",
    higher_better=False,
    scale=1e9,
    unit_label=" \u00d710\u2079 mg",
)

C:\Users\User\AppData\Local\Temp\ipykernel_20384\827036067.py:46: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.savefig(png, dpi=300, bbox_inches="tight")


  Saved: eval_co2_vs_episode.png, eval_co2_vs_episode.pdf
  total_co2: mean=0.00, std=0.00, min=0.00, max=0.00


C:\Users\User\AppData\Local\Temp\ipykernel_20384\827036067.py:47: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.savefig(pdf, bbox_inches="tight")
C:\Users\User\AppData\Local\Temp\ipykernel_20384\827036067.py:47: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.savefig(pdf, bbox_inches="tight")


## 10. Evaluation: Throughput vs Episodes

Throughput (vehicles per hour) during evaluation. Higher is better — indicates the agent moves more vehicles through the network.

In [34]:
plot_eval_metric(
    eval_df, "throughput_per_hour",
    ylabel="Throughput (vehicles/hour)",
    title="MARL DQN Evaluation Throughput vs Episode (Cologne, 120 Episodes)",
    figname="eval_throughput_vs_episode",
    higher_better=True,
)

  Saved: eval_throughput_vs_episode.png, eval_throughput_vs_episode.pdf
  throughput_per_hour: mean=517.08, std=33.76, min=441.00, max=606.00


## 11. Combined Evaluation Dashboard (Reward, Emissions, Throughput)

A single 3-panel figure showing the relationship between reward, emissions, and throughput across evaluation episodes.

In [35]:
eval_panels = [
    ("reward_sum",          "Cumulative Reward",                   "Reward",                     True,  1.0),
    ("total_co2",           "Total CO\u2082 Emissions (\u00d710\u2079 mg)",  "CO\u2082 (\u00d710\u2079 mg)",              False, 1e9),
    ("throughput_per_hour", "Throughput (vehicles/hour)",           "Throughput (veh/hr)",        True,  1.0),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

panel_colors = [COLOR_PRIMARY, COLOR_SECONDARY, COLOR_ACCENT]

for ax, (col, ylabel, short_title, higher_better, scale), color in zip(axes, eval_panels, panel_colors):
    if col not in eval_df.columns:
        ax.text(0.5, 0.5, f"No data for {col}", transform=ax.transAxes, ha="center")
        continue
    y = eval_df[col].dropna()
    if len(y) == 0:
        continue
    eps = eval_df.loc[y.index, "episode"].values
    yv = y.values / scale

    ax.plot(eps, yv, alpha=0.2, color=color, linewidth=0.8)
    w = min(15, max(1, len(yv) // 8))
    ma = pd.Series(yv).rolling(window=w, min_periods=1).mean().values
    ax.plot(eps, ma, color=color, linewidth=2.5, label=f"MA-{w}")
    ax.fill_between(eps, yv, ma, alpha=0.07, color=color)

    mu = np.mean(yv)
    ax.axhline(y=mu, color="gray", linestyle="--", alpha=0.5, linewidth=1)
    ax.annotate(f"mean={mu:.1f}", xy=(eps[-1], mu), fontsize=9, color="gray",
                va="bottom", ha="right")

    ax.set_xlabel("Episode")
    ax.set_ylabel(ylabel)
    ax.set_title(short_title)
    ax.legend(loc="best", fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("MARL DQN Evaluation on Cologne (120 Episodes) — Reward, Emissions & Throughput",
             fontsize=15, y=1.03)
fig.tight_layout()
save_fig("eval_dashboard_3panel")

C:\Users\User\AppData\Local\Temp\ipykernel_20384\1243840471.py:40: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  fig.tight_layout()
C:\Users\User\AppData\Local\Temp\ipykernel_20384\827036067.py:46: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.savefig(png, dpi=300, bbox_inches="tight")
C:\Users\User\AppData\Local\Temp\ipykernel_20384\827036067.py:47: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.savefig(pdf, bbox_inches="tight")


  Saved: eval_dashboard_3panel.png, eval_dashboard_3panel.pdf


C:\Users\User\AppData\Local\Temp\ipykernel_20384\827036067.py:47: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.savefig(pdf, bbox_inches="tight")


## 12. Reward–Emissions–Throughput Correlation

Scatter plots showing pairwise relationships between the three key metrics during evaluation. This helps verify that higher reward correlates with lower emissions and higher throughput.

In [36]:
corr_cols = ["reward_sum", "total_co2", "throughput_per_hour"]
corr_labels = ["Reward", "CO\u2082 (\u00d710\u2079 mg)", "Throughput (veh/hr)"]
corr_scales = [1.0, 1e9, 1.0]

pairs = [(0, 1), (0, 2), (1, 2)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (i, j) in zip(axes, pairs):
    ci, cj = corr_cols[i], corr_cols[j]
    if ci not in eval_df.columns or cj not in eval_df.columns:
        continue
    valid = eval_df[[ci, cj]].dropna()
    if len(valid) == 0:
        continue
    x = valid[ci].values / corr_scales[i]
    y = valid[cj].values / corr_scales[j]

    ax.scatter(x, y, alpha=0.5, s=30, color=COLOR_PRIMARY, edgecolors="white", linewidth=0.3)

    z = np.polyfit(x, y, 1)
    p = np.poly1d(z)
    x_sorted = np.sort(x)
    ax.plot(x_sorted, p(x_sorted), color=COLOR_SMOOTH, linewidth=2, linestyle="--", alpha=0.8)

    from scipy.stats import pearsonr
    r, pval = pearsonr(x, y)
    ax.annotate(f"r = {r:.3f}\np = {pval:.2e}", xy=(0.05, 0.95), xycoords="axes fraction",
                fontsize=10, va="top", ha="left",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

    ax.set_xlabel(corr_labels[i])
    ax.set_ylabel(corr_labels[j])
    ax.grid(True, alpha=0.3)

fig.suptitle("Pairwise Correlation: Reward, Emissions & Throughput (Cologne Evaluation)",
             fontsize=14, y=1.02)
fig.tight_layout()
save_fig("eval_correlation_scatter")

C:\Users\User\AppData\Local\Temp\ipykernel_20384\842448347.py:27: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, pval = pearsonr(x, y)
d:\Anaconda\envs\pytorch_env\lib\site-packages\numpy\lib\_polynomial_impl.py:657: RuntimeWarning: invalid value encountered in divide
  lhs /= scale


LinAlgError: SVD did not converge in Linear Least Squares

## 13. Summary Statistics

In [ ]:
print("=" * 80)
print("TRAINING SUMMARY (Vancouver, 200 Episodes)")
print("=" * 80)
print(f"  Episodes:          {n_episodes}")
print(f"  Total steps:       {checkpoint.get('step_count', 'N/A'):,}")
print(f"  Final epsilon:     {checkpoint.get('epsilon', 'N/A'):.4f}")
print(f"  Reward (all):      mean={np.mean(episode_rewards):.1f}, std={np.std(episode_rewards):.1f}")
print(f"  Reward (last 50):  mean={np.mean(episode_rewards[-50:]):.1f}, std={np.std(episode_rewards[-50:]):.1f}")
print(f"  Best reward:       {np.max(episode_rewards):.1f} @ episode {np.argmax(episode_rewards)+1}")
if len(losses) > 0:
    print(f"  Loss (last 1000):  mean={np.mean(losses[-1000:]):.6f}")

print()
print("=" * 80)
print("EVALUATION SUMMARY (Cologne, 120 Episodes)")
print("=" * 80)
summary_cols = [
    ("reward_sum",          "Reward",           1.0,  ""),
    ("total_co2",           "CO\u2082",              1e9,  " \u00d710\u2079 mg"),
    ("throughput_per_hour", "Throughput",        1.0,  " veh/hr"),
    ("avg_waiting_time",    "Avg Wait Time",    1.0,  " s"),
    ("avg_queue_length",    "Avg Queue Length",  1.0,  " veh"),
    ("avg_speed",           "Avg Speed",        1.0,  " m/s"),
    ("congestion_index",    "Congestion Index", 1.0,  ""),
]
for col, label, scale, unit in summary_cols:
    if col in eval_df.columns:
        vals = eval_df[col].dropna() / scale
        if len(vals) > 0:
            print(f"  {label:20s}: {vals.mean():10.2f} \u00b1 {vals.std():8.2f}{unit}")

print()
print(f"All figures saved to: {FIG_DIR}")
for f in sorted(FIG_DIR.glob("*.png")):
    print(f"  {f.name}")